<a href="https://colab.research.google.com/github/H-S-11/tamil-multiclass-sentiment-analysis/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install -q scikit-learn pandas matplotlib seaborn transformers torch datasets accelerate -q

import re, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, random, torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import warnings


warnings.filterwarnings('ignore')


df = pd.read_csv('PS_train.csv')
print(f"Dataset: {len(df)} rows")


def clean_text(text):
    if pd.isna(text): return ""
    text = re.sub(r'http\S+|@\w+|#\w+', '', str(text))
    text = re.sub(r'[^அ-௺a-zA-Z0-9\s.,!?]', '', text)
    return ' '.join(text.split())


df = df.dropna(subset=['content', 'labels']).drop_duplicates('content')
df['clean_tamil'] = df['content'].apply(clean_text)
df = df[df['clean_tamil'].str.len() > 10]


X = df['clean_tamil'].values
le = LabelEncoder()
y = le.fit_transform(df['labels'])
print("Classes:", dict(zip(range(len(le.classes_)), le.classes_)))


def augment_tamil_data(X, y, n_aug=2):
    augmented_X, augmented_y = [], []

    for text, label in zip(X, y):
        augmented_X.append(text)
        augmented_y.append(label)

        words = text.split()
        if len(words) > 3:
            # Word swap
            idx1, idx2 = random.sample(range(len(words)), 2)
            words[idx1], words[idx2] = words[idx2], words[idx1]
            augmented_X.append(' '.join(words))
            augmented_y.append(label)

            # Remove random word
            if len(words) > 4:
                noise_idx = random.randint(0, len(words)-1)
                words.pop(noise_idx)
                augmented_X.append(' '.join(words))
                augmented_y.append(label)

    return np.array(augmented_X), np.array(augmented_y)


print("Data augmentation:")
X_aug, y_aug = augment_tamil_data(X, y)
print(f"Augmented: {len(X)} → {len(X_aug)} samples")
X_train, X_test, y_train, y_test = train_test_split(X_aug, y_aug, test_size=0.2, random_state=42, stratify=y_aug)


vectorizer = TfidfVectorizer(ngram_range=(1,3), max_features=10000, min_df=2)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


print("Classical ML:")
lr = LogisticRegression(C=3.0, max_iter=5000, class_weight='balanced', random_state=42)
lr.fit(X_train_tfidf, y_train)
lr_pred = lr.predict(X_test_tfidf)
lr_f1 = f1_score(y_test, lr_pred, average='macro')


svm = LinearSVC(C=2.0, class_weight='balanced', max_iter=5000, random_state=42)
svm.fit(X_train_tfidf, y_train)
svm_pred = svm.predict(X_test_tfidf)
svm_f1 = f1_score(y_test, svm_pred, average='macro')


rf = RandomForestClassifier(n_estimators=500, max_depth=25, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_tfidf, y_train)
rf_pred = rf.predict(X_test_tfidf)
rf_f1 = f1_score(y_test, rf_pred, average='macro')


print(f"LR F1: {lr_f1:.3f}, SVM F1: {svm_f1:.3f}, RF F1: {rf_f1:.3f}")


print("XLM-RoBERTa...")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=7)


def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=160)

train_ds = Dataset.from_dict({'text': X_train, 'label': y_train}).map(tokenize, batched=True)
test_ds = Dataset.from_dict({'text': X_test, 'label': y_test}).map(tokenize, batched=True)


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1_macro': f1_score(labels, preds, average='macro')}


training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=300,
    weight_decay=0.2,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    learning_rate=1e-5,
    fp16=torch.cuda.is_available(),
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


trainer.train()
xlmr_results = trainer.evaluate()
xlmr_pred = trainer.predict(test_ds).predictions.argmax(-1)
xlmr_f1 = xlmr_results['eval_f1_macro']


results = {
    'LogisticRegression': {'f1': lr_f1},
    'LinearSVM': {'f1': svm_f1},
    'RandomForest': {'f1': rf_f1},
    'XLM-RoBERTa': {'f1': xlmr_f1}
}


best_model_name = max(results, key=lambda k: results[k]['f1'])
best_f1 = results[best_model_name]['f1']
print(f"\nBEST: {best_model_name} F1={best_f1:.3f}")


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
f1_scores = [results[m]['f1'] for m in results]
axes[0].barh(list(results.keys()), f1_scores)
axes[0].set_title('Model Comparison (Augmented)')
axes[0].set_xlim(0, 1)


cm = confusion_matrix(y_test, xlmr_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[1].set_title('XLM-R Confusion Matrix')


pd.Series(le.inverse_transform(y_test)).value_counts().plot(kind='bar', ax=axes[2])
axes[2].set_title('Test Distribution')
axes[2].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


print("\nClassification Report:")
print(classification_report(y_test, xlmr_pred, target_names=le.classes_))


sample_df = pd.DataFrame({
    'Tamil': X_test[:15],
    'Actual': le.inverse_transform(y_test[:15]),
    'Pred': le.inverse_transform(xlmr_pred[:15])
})


print("\nSample Predictions:")
print(sample_df.to_string(index=False, max_colwidth=40))


results_df = pd.DataFrame({
    'tamil_text': X_test,
    'actual': le.inverse_transform(y_test),
    'predicted': le.inverse_transform(xlmr_pred)
})


results_df.to_csv('PS_predictions.csv', index=False)
print("\nSaved PS_predictions.csv")

**Imports**

In [ ]:
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

**Data Loading**

In [ ]:
print("DATA LOADING")
df = pd.read_csv('PS_train.csv')

print(f"Dataset shape: {df.shape}")

print(f"Dimensions: {df.shape[0]} rows × {df.shape[1]} columns")

print(f"Columns: {df.columns.tolist()}")

print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nFirst 5 rows:")
print(df.head())

print("\nLast 5 rows:")
print(df.tail())

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nDataset info:")
print(df.info())

**Data Cleaning**

In [ ]:
print("DATA CLEANING")

def clean_tamil_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', str(text))
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'[^அ-௺a-zA-Z0-9\s.,!?]', '', text)
    return ' '.join(text.split())

df = df.dropna(subset=['content', 'labels']).drop_duplicates(subset=['content'])
df['clean_text'] = df['content'].apply(clean_tamil_text)
df = df[df['clean_text'].str.len() > 15]
print(f"Clean data shape: {df.shape}")

**Data Augmentation**

In [ ]:
print("DATA AUGMENTATION")
def advanced_tamil_augmentation(X, y, multiplier=3):
    augmented_X, augmented_y = [], []

    for text, label in zip(X, y):
        augmented_X.append(text)
        augmented_y.append(label)

        words = text.split()

        # Word swapping
        if len(words) >= 4:
            for _ in range(multiplier):
                temp_words = words.copy()
                idx1, idx2 = np.random.choice(len(temp_words), 2, replace=False)
                temp_words[idx1], temp_words[idx2] = temp_words[idx2], temp_words[idx1]
                augmented_X.append(' '.join(temp_words))
                augmented_y.append(label)

        # Word deletion
        if len(words) >= 6:
            for _ in range(multiplier//2):
                temp_words = words.copy()
                delete_count = max(1, len(temp_words)//4)
                indices = np.random.choice(len(temp_words), delete_count, replace=False)
                temp_words = [w for i, w in enumerate(temp_words) if i not in indices]
                augmented_X.append(' '.join(temp_words))
                augmented_y.append(label)

    return np.array(augmented_X), np.array(augmented_y)

**Label Encoding, Train-Test Split**

In [ ]:
print("LABEL ENCODING AND SPLIT ")
le = LabelEncoder()
y = le.fit_transform(df['labels'])
X = df['clean_text'].values

print("Class distribution (original):")
class_dist = pd.Series(le.inverse_transform(y)).value_counts().sort_index()
print(class_dist)

# APPLY AUGMENTATION
print("Applying data augmentation...")
X_aug, y_aug = advanced_tamil_augmentation(X, y)
print(f"Augmented: {len(X)} → {len(X_aug)} samples (x{len(X_aug)/len(X):.1f})")

# STRATIFIED SPLIT ON AUGMENTED DATA
X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y_aug, test_size=0.15, random_state=42, stratify=y_aug  # 85/15 split
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

**Feature Engineering**

In [ ]:
print("FEATURE EXTRACTION")
tamil_vectorizer = TfidfVectorizer(
    ngram_range=(1,4),      # Better for Tamil
    max_features=15000,     # More features
    min_df=3,               # Less noise
    max_df=0.8,             # Avoid dominant terms
    sublinear_tf=True       # SVM optimized
)

X_train_tfidf = tamil_vectorizer.fit_transform(X_train)
X_test_tfidf = tamil_vectorizer.transform(X_test)
print(f"Feature matrix shape: {X_train_tfidf.shape}")

**Model Training with Cross Validation**

In [ ]:
print("MODEL TRAINING")
print("_" * 80)

# Linear SVM (Primary model)
svm_model = LinearSVC(
    C=1.8,
    class_weight='balanced',
    max_iter=10000,
    random_state=42,
    dual=False
)

# Cross-validation
cv_scores = cross_val_score(svm_model, X_train_tfidf, y_train,
                           cv=5, scoring='f1_macro', n_jobs=-1)
print(f"SVM CV F1 scores: {cv_scores}")
print(f"SVM CV Mean F1: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

svm_model.fit(X_train_tfidf, y_train)
svm_pred = svm_model.predict(X_test_tfidf)

**Evaluation Metrics**

In [ ]:
print("EVALUATION METRICS")
print("_" * 80)

svm_accuracy = accuracy_score(y_test, svm_pred)
svm_f1_macro = f1_score(y_test, svm_pred, average='macro')
svm_f1_weighted = f1_score(y_test, svm_pred, average='weighted')

metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Macro F1', 'Weighted F1'],
    'Score': [svm_accuracy, svm_f1_macro, svm_f1_weighted]
})
print(metrics_df.round(4))

**Visualisation**

In [ ]:
print("VISUALIZATIONS")
print("_" * 80)

# Create larger individual plots (one by one)
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('TAMIL TEXT CLASSIFICATION - VISUALIZATION DASHBOARD', fontsize=16, fontweight='bold')

# 1. CROSS-VALIDATION HISTORY
ax1 = plt.subplot(2, 2, 1)
ax1.plot(range(1, 6), cv_scores, 'bo-', linewidth=3, markersize=12, markerfacecolor='darkblue')
ax1.set_title('Cross-Validation F1 Scores (5-Fold)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Fold', fontsize=12)
ax1.set_ylabel('Macro F1 Score', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

for i, score in enumerate(cv_scores, 1):
    ax1.annotate(f'{score:.3f}', (i, score), textcoords="offset points",
                xytext=(0,10), ha='center', fontsize=11)

# 2. MODEL PERFORMANCE COMPARISON
ax2 = plt.subplot(2, 2, 2)
comparison_models = {
    'LinearSVM': svm_f1_macro,
    'Baseline': 0.40
}

bars = ax2.bar(comparison_models.keys(), comparison_models.values(),
               color=['darkblue', 'lightgray'], alpha=0.8, edgecolor='black', linewidth=2)
ax2.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax2.set_ylabel('Macro F1 Score', fontsize=12)
ax2.set_ylim(0, 1)

for bar, val in zip(bars, comparison_models.values()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
             f'{val:.3f}', ha='center', va='bottom', fontsize=13, fontweight='bold')

# 3. CONFUSION MATRIX
ax3 = plt.subplot(2, 2, 3)
cm = confusion_matrix(y_test, svm_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            annot_kws={'size': 12, 'weight': 'bold'},
            cbar_kws={'label': 'Count', 'shrink': 0.8})

ax3.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax3.set_xlabel('Predicted Label', fontsize=12)
ax3.set_ylabel('True Label', fontsize=12)

# 4. PER-CLASS F1 SCORES
ax4 = plt.subplot(2, 2, 4)
report_dict = classification_report(y_test, svm_pred,
                                  target_names=le.classes_,
                                  output_dict=True)

f1_scores = [report_dict[cls]['f1-score'] for cls in le.classes_]
colors = ['red' if score < 0.5 else 'green' for score in f1_scores]
bars = ax4.bar(le.classes_, f1_scores, color=colors, alpha=0.8,
               edgecolor='black', linewidth=1.5)

ax4.set_title('Per-Class F1 Scores', fontsize=14, fontweight='bold')
ax4.set_ylabel('F1 Score', fontsize=12)
ax4.set_ylim(0, 1)
ax4.tick_params(axis='x', rotation=45)

for bar, score in zip(bars, f1_scores):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{score:.2f}', ha='center', va='bottom',
             fontsize=11, fontweight='bold')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

print("\nVisualization Summary:")
print("_" * 80)
print(f"CV Mean F1:  {cv_scores.mean():.3f}")
print(f"Test F1:     {svm_f1_macro:.3f}")
print(f"Improvement: {((svm_f1_macro - 0.27)/0.27*100):.1f}%")

**Classification Report**

In [ ]:
print("CLASSIFICATION REPORT")
print("_" * 80)
print(classification_report(y_test, svm_pred, target_names=le.classes_))

**Sample Predictions**

In [ ]:
print("SAMPLE PREDICTIONS (Top 15)")
print("_" * 80)

# Set global display options for clean output
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', None)

sample_df = pd.DataFrame({
    'ID': range(1, 16),
    'Tamil_Text': X_test[:15],
    'True_Label': le.inverse_transform(y_test[:15]),
    'Predicted': le.inverse_transform(svm_pred[:15]),
    'Correct': y_test[:15] == svm_pred[:15]
})

print("\nModel Summary:")
print("_" * 80)
print(sample_df[['ID', 'True_Label', 'Predicted', 'Correct']].to_string(
    index=False,
    max_colwidth=20
))

print("\nPREDICTION STATISTICS:")
print("_" * 80)
correct_count = sum(sample_df['Correct'])
print(f"Correct predictions: {correct_count}/15 ({correct_count/15*100:.1f}%)")
print(f"Incorrect predictions: {15-correct_count}/15 ({(15-correct_count)/15*100:.1f}%)")

# Reset display options
pd.reset_option('display.max_colwidth')

**Model Summary**

In [ ]:
print("SUMMARY")
print("_" * 80)

print(f"Dataset: {len(df):,} clean samples")
print(f"Features: {X_train_tfidf.shape[1]:,} TF-IDF features")
print(f"Test Accuracy: {svm_accuracy:.4f}")
print(f"Test Macro F1:  {svm_f1_macro:.4f}")
print(f"Cross-val F1:   {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
print(f"Improvement:    {((svm_f1_macro - 0.27)/0.27 * 100):.1f}% over baseline")

**Save Results**

In [ ]:
print("SAVING RESULTS")

results_df = pd.DataFrame({
    'tamil_text': X_test,
    'true_label': le.inverse_transform(y_test),
    'predicted_label': le.inverse_transform(svm_pred),
    'is_correct': (y_test == svm_pred).astype(int),
    'confidence': np.max(svm_model.decision_function(X_test_tfidf), axis=1)
})

results_df.to_csv('PS_predictions_SVM.csv', index=False)
print("Saved: PS_predictions_SVM.csv")

print("\n" + "_" * 80)
print("Successfully Executed")
print("_" * 80)